In [41]:
import pandas as pd

df = pd.read_excel("perfect_final_data_for_multilevel_analysis.xlsx")

# keep only measured BMI
df = df[df["V447"] == 0].copy()

# drop measurement column
df = df.drop(columns=["V447"])

print(df.shape)

(8502, 40)


In [42]:
feature_cols = [
    'V001','V012', 'V024', 'V025', 'V106','V404','V502','V501','V714',
    'Drinking_water', 'Religion_status', 'Media_acccess',
    'Wealth_Status', 'Number_of_children', 'Living_children',
    'Birth_Interval', 'contraceptive_use',
    'husband_education', 'profession_of_husband',
    'Age_of_husband', 'Respondent_Age_first_birth'
]

X = df[feature_cols].copy()
y = df["BMI"].copy()

In [43]:
print(y.value_counts())

BMI
2    4469
3    2533
1     806
4     694
Name: count, dtype: int64


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

nominal_cols = [
    "V024",               # division
    "Drinking_water",
    "profession_of_husband"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_cols)
    ],
    remainder="passthrough"
)

In [46]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [47]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function='MultiClass',
    verbose=200,
    random_state=42
)

pipeline2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [48]:
y_train_mapped = y_train.map({1:0,2:1,3:2,4:3})
y_test_mapped = y_test.map({1:0,2:1,3:2,4:3})

In [49]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(sampling_strategy='not majority', random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train_mapped)

In [50]:
# pipeline.fit(X_train, y_train_mapped)
pipeline.fit(X_train_res, y_train_res)

C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['V024', 'Drinking_water',
                                                   'profession_of_husband'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_st...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.02,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=1000, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [51]:
pipeline2.fit(X_train_res, y_train_res)

0:	learn: 1.3796279	total: 29.8ms	remaining: 29.7s
200:	learn: 1.0881307	total: 3.07s	remaining: 12.2s
400:	learn: 0.9738262	total: 6.33s	remaining: 9.46s
600:	learn: 0.8839806	total: 9.4s	remaining: 6.24s
800:	learn: 0.8114384	total: 12.6s	remaining: 3.14s
999:	learn: 0.7502058	total: 15.8s	remaining: 0us


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['V024', 'Drinking_water',
                                                   'profession_of_husband'])])),
                ('classifier',
                 CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.03, loss_function='MultiClass', random_state=42, verbose=200))])

In [52]:
y_pred_mapped = pipeline.predict(X_test)

inverse_map = {0:1,1:2,2:3,3:4}
y_pred = pd.Series(y_pred_mapped).map(inverse_map)

In [53]:
y_pred_mapped2 = pipeline.predict(X_test)

inverse_map2 = {0:1,1:2,2:3,3:4}
y_pred2 = pd.Series(y_pred_mapped2).map(inverse_map2)

In [54]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.4191651969429747
              precision    recall  f1-score   support

           1       0.19      0.23      0.21       161
           2       0.57      0.49      0.53       894
           3       0.38      0.39      0.38       507
           4       0.18      0.27      0.22       139

    accuracy                           0.42      1701
   macro avg       0.33      0.35      0.33      1701
weighted avg       0.45      0.42      0.43      1701



In [55]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred2))
print(classification_report(y_test, y_pred2))

Accuracy: 0.4191651969429747
              precision    recall  f1-score   support

           1       0.19      0.23      0.21       161
           2       0.57      0.49      0.53       894
           3       0.38      0.39      0.38       507
           4       0.18      0.27      0.22       139

    accuracy                           0.42      1701
   macro avg       0.33      0.35      0.33      1701
weighted avg       0.45      0.42      0.43      1701

